# Curious Minds Club Chatbot
A topic-locked chatbot for an after-school club. Conversations are saved in Supabase and shown in a Gradio interface.

## Setup

In [ ]:
!pip -q install langgraph langchain langchain-openai langgraph-checkpoint-postgres "psycopg[binary]" gradio

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Secrets
In Colab, add these two secrets and turn on **Notebook access** for both: `OPENAI_API_KEY` and `SUPABASE_DB_URL`. Use the Supabase **Session pooler** URI (IPv4); Colab cannot reach the IPv6 direct connection. The values are loaded from Colab Secrets and are never displayed.

## Model

In [ ]:
MODEL_NAME = "openai:gpt-6-luna"

In [ ]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(MODEL_NAME)

## Topics and system message

In [ ]:
from langchain_core.messages import SystemMessage

TOPICS = ["Astronomy", "Cooking", "Football", "World Geography", "Python Programming"]
DEFAULT_TOPIC = "Astronomy"

SYSTEM_TEMPLATE = """You are the friendly chatbot of the {topic} club at Curious Minds, an after-school enrichment center for teenagers.

The current club is {topic}. Answer only questions about {topic}. Keep answers short, clear and suitable for teenagers.

Earlier messages in this conversation may belong to a different club. Only the current club counts: never continue or answer an earlier topic, even if it appears in the history.

If a question is not about {topic}, including a follow-up about an earlier club, do not answer it. Instead, say kindly that you can't help with that in the {topic} club, and suggest switching to the club that fits the question, if there is one. The clubs are: {clubs}.

Do not write essays, homework or assignments for other subjects, even if the request mentions {topic}.
"""

def build_system_message(topic: str) -> SystemMessage:
    """Build the current club's system message."""
    return SystemMessage(SYSTEM_TEMPLATE.format(topic=topic, clubs=", ".join(TOPICS)))  # <-- the topic enters the system message here

## State

In [ ]:
from typing import Annotated, TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
    topic: str

## Node

In [ ]:
def chatbot(state: State) -> dict:
    """Answer once using the currently selected club topic."""
    system = build_system_message(state["topic"])
    reply = llm.invoke([system] + state["messages"])
    return {"messages": [reply]}

## Graph

In [ ]:
from langgraph.graph import END, START, StateGraph

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

## Checkpointer
The graph code is the same as with `InMemorySaver`; only the checkpointer line changes. Supabase stores conversation state in Postgres so it survives a Colab restart.

In [ ]:
import os, psycopg
from psycopg.rows import dict_row
from google.colab import userdata
from langgraph.checkpoint.postgres import PostgresSaver

# OPENAI_API_KEY is already set in the Secrets cell at the top of the notebook
conn = psycopg.connect(userdata.get("SUPABASE_DB_URL"),
                       autocommit=True, row_factory=dict_row, prepare_threshold=0)
checkpointer = PostgresSaver(conn)  # <-- the checkpointer is created here
checkpointer.setup()  # creates the checkpoint tables (safe to run again)
graph = builder.compile(checkpointer=checkpointer)

## Drawing

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

## Graph test

In [ ]:
config = {"configurable": {"thread_id": "test-1"}}

def ask(text, topic):
    result = graph.invoke({"messages": [HumanMessage(text)], "topic": topic}, config)
    print(f"[{topic}] {text}\n -> {result['messages'][-1].content}\n")

ask("How do I make pancakes?", "Cooking")                # answers
ask("Who won the 2022 World Cup?", "Cooking")            # refuses, suggests the Football club
ask("Who won the 2022 World Cup?", "Football")           # answers
ask("And how long do I fry each pancake?", "Football")   # refuses: old topic

In [ ]:
values = graph.get_state(config).values
print(f"Messages: {len(values['messages'])}")
print(f"Topic: {values['topic']}")

## Conversations

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage

def list_conversations():
    """Read saved conversation labels from the checkpointer."""
    conversations = []
    thread_ids = []
    seen = set()
    # Keep these steps separate: checkpointer.list holds the connection lock while yielding.
    for checkpoint in checkpointer.list(None):
        thread_id = checkpoint.config["configurable"]["thread_id"]
        if thread_id in seen:
            continue
        seen.add(thread_id)
        thread_ids.append(thread_id)

    for thread_id in thread_ids:
        state = graph.get_state({"configurable": {"thread_id": thread_id}}).values
        messages = state.get("messages", [])
        first_human = next((m for m in messages if isinstance(m, HumanMessage)), None)
        if first_human is None:
            continue
        words = first_human.content.split()
        preview = " ".join(words)
        if len(preview) > 40:
            preview = preview[:39].rstrip() + "…"
        conversations.append((f"[{state.get('topic', DEFAULT_TOPIC)}] {preview}", thread_id))
    return conversations

def load_conversation(thread_id):
    """Load saved Human and AI messages and the conversation topic."""
    state = graph.get_state({"configurable": {"thread_id": thread_id}}).values
    chat = [
        {"role": "user" if isinstance(message, HumanMessage) else "assistant", "content": message.content}
        for message in state.get("messages", [])
        if isinstance(message, (HumanMessage, AIMessage))
    ]
    return chat, state.get("topic", DEFAULT_TOPIC)  # <-- loads an old conversation back into the interface

## Interface

In [ ]:
import gradio as gr
import uuid

def new_thread():
    """Create a fresh conversation identifier."""
    return str(uuid.uuid4())

def refresh_conversations():
    """Refresh dropdown options from the database."""
    return gr.Dropdown(choices=list_conversations())

def open_saved_conversation(thread_id):
    """Load a saved conversation into the chat interface."""
    if not thread_id:
        return [], DEFAULT_TOPIC, thread_id
    messages, topic = load_conversation(thread_id)
    return messages, topic, thread_id

def start_new_conversation(topic):
    """Start a new thread and clear the open chat."""
    return [], None, new_thread()

def send_message(text, topic, thread_id):
    """Send one new message and rebuild the visible transcript."""
    if not text or not text.strip():
        return gr.skip(), "", gr.skip(), gr.skip()
    config = {"configurable": {"thread_id": thread_id}}
    graph.invoke({"messages": [HumanMessage(text)], "topic": topic}, config)
    messages = graph.get_state(config).values["messages"]
    chat = [
        {"role": "user" if isinstance(message, HumanMessage) else "assistant", "content": message.content}
        for message in messages
        if isinstance(message, (HumanMessage, AIMessage))
    ]
    return chat, "", gr.Dropdown(choices=list_conversations(), value=thread_id), thread_id

with gr.Blocks(title="Curious Minds Club Chatbot") as demo:
    gr.Markdown("# Curious Minds Club Chatbot\nPick a club; the bot only talks about that club. Conversations are saved.")
    with gr.Row():
        club = gr.Dropdown(TOPICS, value=DEFAULT_TOPIC, label="Club")
        saved = gr.Dropdown(choices=list_conversations(), label="Saved conversations")
    with gr.Row():
        refresh = gr.Button("Refresh list")
        new = gr.Button("New conversation")
    chat = gr.Chatbot(type="messages", label="Chat window")
    with gr.Row():
        message = gr.Textbox(label="Message", placeholder="Ask a question about this club", scale=4)
        send = gr.Button("Send", variant="primary")
    gr.Examples(
        examples=[["What are the moons of Mars?"], ["How do I make pancakes?"], ["Who won the 2022 World Cup?"]],
        inputs=message,
    )
    thread_id = gr.State()

    demo.load(lambda: (new_thread(), gr.Dropdown(choices=list_conversations())), outputs=[thread_id, saved])
    refresh.click(refresh_conversations, outputs=saved)
    new.click(start_new_conversation, inputs=club, outputs=[chat, saved, thread_id])
    saved.change(open_saved_conversation, inputs=saved, outputs=[chat, club, thread_id])
    send.click(send_message, inputs=[message, club, thread_id], outputs=[chat, message, saved, thread_id])
    message.submit(send_message, inputs=[message, club, thread_id], outputs=[chat, message, saved, thread_id])

demo.launch()

When you launch the interface, the printed `gradio.live` link is public and uses your OpenAI key. Do not share it, and stop this cell when you are done.